# 1. Rename the image in image folder

In [5]:
import os
import xml.etree.ElementTree as ET

# Define paths
opf_path = r'C:\Users\kevin\Downloads\test_book\深\OEBPS\content.opf'
image_dir = r'C:/Users/kevin/Downloads/test_book/image'

# Define the OPF namespace
opf_namespace = {'opf': 'http://www.idpf.org/2007/opf'}

# Parse the content.opf file
try:
    tree = ET.parse(opf_path)
    root = tree.getroot()

    # Find all image items in order using the namespace
    images = []
    for item in root.findall('.//opf:item', opf_namespace):
        if item.get('media-type') == 'image/jpeg':
            href = item.get('href')
            if href:
                images.append(href)

    print(f"Found {len(images)} JPEG images in content.opf: {images}")

    # Check if the image directory exists
    if not os.path.exists(image_dir):
        raise FileNotFoundError(f"Image directory not found: {image_dir}")

    # Rename the files in the image folder
    if images:
        # Check if all image files exist
        missing_files = []
        for href in images:
            if not os.path.exists(os.path.join(image_dir, href)):
                missing_files.append(href)
        if missing_files:
            print(f"Error: The following image files are missing in {image_dir}: {missing_files}")
            exit(1)

        # First image is the cover
        cover_href = images[0]
        cover_path = os.path.join(image_dir, cover_href)
        new_cover_path = os.path.join(image_dir, 'cover.jpg')
        print(f"Renaming {cover_path} to {new_cover_path}")
        os.rename(cover_path, new_cover_path)
        
        # Rename the rest sequentially
        for i, href in enumerate(images[1:], start=1):
            new_name = f'i-{i:03d}.jpg'
            old_path = os.path.join(image_dir, href)
            new_path = os.path.join(image_dir, new_name)
            print(f"Renaming {old_path} to {new_path}")
            os.rename(old_path, new_path)

        print("Renaming completed successfully.")
    else:
        print("No JPEG images found in content.opf.")

except FileNotFoundError as e:
    print(f"Error: {str(e)}")
except ET.ParseError:
    print(f"Error: Failed to parse {opf_path}. Check if the file is valid XML.")
except Exception as e:
    print(f"An error occurred: {str(e)}")

Found 171 JPEG images in content.opf: ['image_rsrc18F.jpg', 'image_rsrc18H.jpg', 'image_rsrc18K.jpg', 'image_rsrc18N.jpg', 'image_rsrc18R.jpg', 'image_rsrc18T.jpg', 'image_rsrc18V.jpg', 'image_rsrc18X.jpg', 'image_rsrc18Z.jpg', 'image_rsrc191.jpg', 'image_rsrc193.jpg', 'image_rsrc195.jpg', 'image_rsrc197.jpg', 'image_rsrc199.jpg', 'image_rsrc19B.jpg', 'image_rsrc19D.jpg', 'image_rsrc19F.jpg', 'image_rsrc19H.jpg', 'image_rsrc19K.jpg', 'image_rsrc19N.jpg', 'image_rsrc19R.jpg', 'image_rsrc19T.jpg', 'image_rsrc19V.jpg', 'image_rsrc19X.jpg', 'image_rsrc19Z.jpg', 'image_rsrc1A1.jpg', 'image_rsrc1A3.jpg', 'image_rsrc1A5.jpg', 'image_rsrc1A7.jpg', 'image_rsrc1A9.jpg', 'image_rsrc1AB.jpg', 'image_rsrc1AD.jpg', 'image_rsrc1AF.jpg', 'image_rsrc1AH.jpg', 'image_rsrc1AK.jpg', 'image_rsrc1AN.jpg', 'image_rsrc1AR.jpg', 'image_rsrc1AT.jpg', 'image_rsrc1AV.jpg', 'image_rsrc1AX.jpg', 'image_rsrc1AZ.jpg', 'image_rsrc1B1.jpg', 'image_rsrc1B3.jpg', 'image_rsrc1B5.jpg', 'image_rsrc1B7.jpg', 'image_rsrc1B9.j

# 2. Modified EPub 

In [ ]:
import os
import xml.etree.ElementTree as ET

# Hardcoded template based on provided p-195.xhtml
XHTML_TEMPLATE = '''<?xml version="1.0" encoding="UTF-8"?><!DOCTYPE html><html xmlns="http://www.w3.org/1999/xhtml">
<head>
<meta name="viewport" content="width=1066, height=1600"/>
<title>膽大黨 7</title>
<link rel="stylesheet" type="text/css" href="../style/fixed-layout-jp.css"/>

<!-- kobo-style -->
<script xmlns="http://www.w3.org/1999/xhtml" type="text/javascript" src="../../js/kobo.js"/>

</head>
<body>
<div>
<svg xmlns="http://www.w3.org/2000/svg" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:ev="http://www.w3.org/2001/xml-events" version="1.1" baseProfile="full" viewBox="0 0 1066 1600">
<image xlink:href="../image/i-{num:03d}.jpg" x="0" y="0" width="1066" height="1600"/></svg>
</div>
</body>
</html>'''

def main():
    # Ask user for the path to the item folder
    item_path = input("Enter the path to the item folder: ").strip()
    
    if not os.path.isdir(item_path):
        print("Error: Provided path is not a valid directory.")
        return
    
    image_dir = os.path.join(item_path, 'image')
    xhtml_dir = os.path.join(item_path, 'xhtml')
    opf_path = os.path.join(item_path, 'standard.opf')
    
    if not os.path.isdir(image_dir):
        print("Error: 'image' folder not found.")
        return
    if not os.path.isdir(xhtml_dir):
        print("Error: 'xhtml' folder not found.")
        return
    if not os.path.isfile(opf_path):
        print("Error: 'standard.opf' not found.")
        return
    
    # Find max image number
    image_files = [f for f in os.listdir(image_dir) if f.startswith('i-') and f.endswith('.jpg')]
    image_nums = []
    for f in image_files:
        try:
            dot_pos = f.rfind('.')
            num_str = f[2:dot_pos]
            num = int(num_str)
            image_nums.append(num)
        except ValueError:
            pass
    max_image = max(image_nums) if image_nums else 0
    
    # Find max xhtml number
    xhtml_files = [f for f in os.listdir(xhtml_dir) if f.startswith('p-') and f.endswith('.xhtml')]
    xhtml_nums = []
    for f in xhtml_files:
        try:
            dot_pos = f.rfind('.')
            num_str = f[2:dot_pos]
            num = int(num_str)
            xhtml_nums.append(num)
        except ValueError:
            pass
    max_xhtml = max(xhtml_nums) if xhtml_nums else 0
    
    print(f"Max image number: {max_image}")
    print(f"Max xhtml number: {max_xhtml}")
    
    if max_image <= max_xhtml:
        print("No new XHTML files needed.")
        return
    
    # Create new XHTML files
    for num in range(max_xhtml + 1, max_image + 1):
        new_xhtml_path = os.path.join(xhtml_dir, f'p-{num:03d}.xhtml')
        with open(new_xhtml_path, 'w', encoding='utf-8') as f:
            f.write(XHTML_TEMPLATE.format(num=num))
        print(f"Created: {new_xhtml_path}")
    
    # Parse OPF to get existing data
    ns = {'opf': 'http://www.idpf.org/2007/opf'}
    tree = ET.parse(opf_path)
    root = tree.getroot()
    
    manifest = root.find('opf:manifest', ns)
    if manifest is None:
        print("Error: No <manifest> found in OPF.")
        return
    
    spine = root.find('opf:spine', ns)
    if spine is None:
        print("Error: No <spine> found in OPF.")
        return
    
    # Collect existing item hrefs
    existing_hrefs = [item.get('href') for item in manifest.findall('opf:item', ns) if item.get('href')]
    
    # Get current max id number and padding
    id_nums = []
    padding = 0
    for item in manifest.findall('opf:item', ns):
        item_id = item.get('id')
        if item_id and item_id.startswith('id'):
            digit_part = item_id[2:]
            if digit_part.isdigit():
                num = int(digit_part)
                id_nums.append(num)
                padding = max(padding, len(digit_part))
    
    max_id = max(id_nums) if id_nums else 0
    if padding == 0:
        padding = 4  # Default padding to 4 digits
    
    # Determine starting property for spine by checking the last itemref
    itemrefs = spine.findall('opf:itemref', ns)
    if itemrefs:
        last_prop = itemrefs[-1].get('properties')
        next_prop = 'page-spread-left' if last_prop == 'page-spread-right' else 'page-spread-right'
    else:
        next_prop = 'page-spread-left'  # Default start
    
    # Prepare new lines for manifest and spine
    new_manifest_lines = []
    new_spine_lines = []
    current_id = max_id + 1
    
    for num in range(max_xhtml + 1, max_image + 1):
        num_str = f'{num:03d}'
        
        # Image item if not existing
        img_href = f'image/i-{num_str}.jpg'
        if img_href not in existing_hrefs:
            img_id = f'id{current_id:0{padding}d}'
            new_manifest_lines.append(f'<item id="{img_id}" href="{img_href}" media-type="image/jpeg"/>')
            print(f"Prepared image for manifest: {img_id}")
            current_id += 1
        
        # XHTML item
        xhtml_href = f'xhtml/p-{num_str}.xhtml'
        if xhtml_href not in existing_hrefs:
            xhtml_id = f'id{current_id:0{padding}d}'
            new_manifest_lines.append(f'<item id="{xhtml_id}" href="{xhtml_href}" media-type="application/xhtml+xml" properties="svg"/>')
            print(f"Prepared XHTML for manifest: {xhtml_id}")
            current_id += 1
        
        # Spine itemref
        prop = next_prop
        new_spine_lines.append(f'<itemref idref="{xhtml_id}" properties="{prop}"/>')
        print(f"Prepared for spine: {xhtml_id} with {prop}")
        next_prop = 'page-spread-left' if prop == 'page-spread-right' else 'page-spread-right'
    
    # Now, update the OPF file by inserting lines as text to preserve formatting
    with open(opf_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    # Find positions for </manifest> and </spine>
    manifest_end = None
    spine_end = None
    for i, line in enumerate(lines):
        if '</manifest>' in line.strip():
            manifest_end = i
        if '</spine>' in line.strip():
            spine_end = i
    
    if manifest_end is None or spine_end is None:
        print("Error: Could not find </manifest> or </spine> in OPF.")
        return
    
    # Detect indent for manifest items
    manifest_indent = '    '  # Default
    for j in range(manifest_end - 1, -1, -1):
        stripped = lines[j].strip()
        if stripped.startswith('<item '):
            manifest_indent = lines[j][:lines[j].find('<')]
            break
    
    # Detect indent for spine itemrefs
    spine_indent = '    '  # Default
    for j in range(spine_end - 1, -1, -1):
        stripped = lines[j].strip()
        if stripped.startswith('<itemref '):
            spine_indent = lines[j][:lines[j].find('<')]
            break
    
    # Add indents and newlines
    new_manifest_lines = [manifest_indent + l + '\n' for l in new_manifest_lines]
    new_spine_lines = [spine_indent + l + '\n' for l in new_spine_lines]
    
    # Insert into lines (insert into spine first since it's after manifest)
    lines = lines[:spine_end] + new_spine_lines + lines[spine_end:]
    lines = lines[:manifest_end] + new_manifest_lines + lines[manifest_end:]
    
    # Write back
    with open(opf_path, 'w', encoding='utf-8') as f:
        f.writelines(lines)
    
    print("Updated standard.opf successfully while preserving formatting.")

if __name__ == "__main__":
    main()

# Compile to Epub

In [ ]:
import os
import zipfile

def create_epub_from_folder(folder_path, output_epub_path):
    # Ensure the folder path ends with a separator
    folder_path = os.path.abspath(folder_path)
    
    # Create a ZIP file (EPUB is essentially a ZIP archive)
    with zipfile.ZipFile(output_epub_path, 'w', zipfile.ZIP_DEFLATED) as epub_file:
        # Walk through the directory
        for root, dirs, files in os.walk(folder_path):
            for file in files:
                file_path = os.path.join(root, file)
                # Calculate the relative path inside the ZIP
                arcname = os.path.relpath(file_path, folder_path)
                epub_file.write(file_path, arcname)

if __name__ == '__main__':
    # Specify the folder containing your HTML files and resources
    input_folder = '/Users/kongyeekwok/Downloads/膽大黨 7_modified'  # Replace with your folder path
    output_epub = '/Users/kongyeekwok/Downloads/Desktop/eBook list/魔都精兵18.epub'  # Replace with desired output EPUB file path
    create_epub_from_folder(input_folder, output_epub)
    print(f"EPUB file created at {output_epub}")